# Часть 2. Предобработка данных

**Датасет:** Student Performance Factors (Kaggle, lainguyn123)

В этом ноутбуке датасет загружается заново, очищается от дубликатов и логических ошибок, сохраняются строки с пропусками, затем данные разделяются на train/test и выполняется предобработка признаков.


In [ ]:
!pip install -q opendatasets legacy-cgi
import opendatasets as od
od.download("https://www.kaggle.com/datasets/lainguyn123/student-performance-factors")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
PATH = "/content/student-performance-factors/StudentPerformanceFactors.csv"
df = pd.read_csv(PATH)
rows_raw = len(df)
print("Исходный размер:", df.shape)
df.head()

## 1 Очистка датасета
Сначала чистим, потом анализируем — иначе любые средние и корреляции будут посчитаны на мусоре.

### 1.1 Единообразие текстовых значений
 Сначала смотрим на уникальные значения, потом обрезаем пробелы.

In [ ]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    print(col, "->", sorted(df[col].dropna().unique()))

In [ ]:
for col in cat_cols:
    df[col] = df[col].str.strip()   # убираем пробелы по краям, регистр не трогаем — он здесь единообразный

### 1.2 Дубликаты
Одна строка — один студент. Если совпали все 20 столбцов сразу, это почти наверняка повторная загрузка одной записи, а не два разных студента с одинаковыми показателями. Оставляем первое вхождение, остальные убираем.

In [ ]:
dups = int(df.duplicated().sum())
print("Полных дублей:", dups)
df = df.drop_duplicates().reset_index(drop=True)
rows_after_dups = len(df)

### 1.3 Логические ошибки
Здесь проверяем, что значения вообще возможны:
- баллы и посещаемость — от 0 до 100;
- сон — от 0 до 24 часов в сутки;
- часы учёбы, спорта — не отрицательные и не больше 168 (столько часов в неделе);
- совместное правило: сон за неделю (часы в сутки × 7) + учёба + физическая активность не могут в сумме превышать 168 часов.

Строки, где правило нарушено, удаляем целиком. Правильное значение мы всё равно не знаем, а таких строк, скорее всего, единицы, так что потеря несущественна. Особенно важно, если ошибка в целевой переменной: подставлять "что-нибудь похожее" в `Exam_Score` нельзя.

In [ ]:
rules = {
    "Exam_Score":        (0, 100),
    "Attendance":        (0, 100),
    "Previous_Scores":   (0, 100),
    "Sleep_Hours":       (0, 24),
    "Hours_Studied":     (0, 168),
    "Physical_Activity": (0, 168),
    "Tutoring_Sessions": (0, 168),
}

bad = pd.Series(False, index=df.index)
report = []
for col, (lo, hi) in rules.items():
    mask = (df[col] < lo) | (df[col] > hi)
    report.append({"Правило": f"{col} вне [{lo}; {hi}]", "Нарушений": int(mask.sum())})
    bad |= mask.fillna(False)

week_total = df["Sleep_Hours"] * 7 + df["Hours_Studied"] + df["Physical_Activity"]
mask_week = week_total > 168
report.append({"Правило": "сон*7 + учёба + спорт > 168 ч/нед", "Нарушений": int(mask_week.sum())})
bad |= mask_week.fillna(False)

pd.DataFrame(report)

In [ ]:
print("Строк с логическими ошибками:", int(bad.sum()))
df[bad]   # посмотрим на них глазами перед удалением

In [ ]:
df = df[~bad].reset_index(drop=True)
rows_after_logic = len(df)

### 1.4 Пропуски
Смотрим, где пропуски и какая это доля.

In [ ]:
miss = pd.DataFrame({"Пропусков": df.isna().sum(), "Доля, %": (df.isna().mean() * 100).round(2)})
miss[miss["Пропусков"] > 0]

**Что делаем и почему.** Пропуски находятся в категориальных столбцах. Вместо удаления строк мы сохраняем наблюдения: после разделения на train/test заполняем пропуски наиболее частой категорией, используя `SimpleImputer(strategy="most_frequent")`. Импьютер обучается только на train, а затем тем же правилом преобразует test. Это сохраняет больше данных и предотвращает утечку статистики из тестовой выборки.


In [ ]:
# Строки с пропусками не удаляем: заполнение будет обучено на train ниже.
rows_final = len(df)
print("Строк сохранено:", rows_final)
print("Пропусков перед разделением:", int(df.isna().sum().sum()))


### 1.5 Итог очистки

In [ ]:
summary = pd.DataFrame({
    "Шаг": ["Исходный датасет", "После удаления дублей", "После удаления логических ошибок", "После решения по пропускам (строки сохранены)"],
    "Строк": [rows_raw, rows_after_dups, rows_after_logic, rows_final],
})
summary["Потеряно на шаге"] = (-summary["Строк"].diff()).fillna(0).astype(int)
print(f"Всего удалено на этапах дубликатов и логических ошибок: {rows_raw - rows_after_logic} строк")
print("Пропусков перед split:", int(df.isna().sum().sum()), "| дублей осталось:", int(df.duplicated().sum()))
summary


## 2 Предобработка
Сначала делим данные на train и test (80/20), и только потом подбираем любые параметры преобразований — строго по train. Иначе в расчёт среднего, минимума и т. п. попадут данные test, а это утечка: модель "подсмотрит" в тест ещё до обучения.

In [ ]:
X = df.drop(columns="Exam_Score")
y = df["Exam_Score"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("train:", X_train.shape, "| test:", X_test.shape)

### 2.1 Масштабирование одного числового признака
Берём `Hours_Studied` — у него есть правый "хвост" (редкие студенты, которые учатся очень много), поэтому методы должны вести себя по-разному. Применяем четыре метода: StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler. Каждый обучаем на train и смотрим на распределение признака на train до и после.

In [ ]:
feature = "Hours_Studied"
x_train = X_train[[feature]]

scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler":   MinMaxScaler(),
    "MaxAbsScaler":   MaxAbsScaler(),
    "RobustScaler":   RobustScaler(),
}

scaled = {"Исходный признак": x_train[feature].values}
for name, sc in scalers.items():
    scaled[name] = sc.fit_transform(x_train).ravel()   # fit — только на train

stats_table = pd.DataFrame({
    name: {"среднее": v.mean(), "std": v.std(), "медиана": np.median(v), "min": v.min(), "max": v.max()}
    for name, v in scaled.items()
}).T.round(3)
stats_table

In [ ]:
# Гистограммы распределения до и после масштабирования — отдельно и крупно
plot_items = [("Исходный признак", X_train[feature].values)] + list(scaled.items())
fig, axes = plt.subplots(len(plot_items), 1, figsize=(11, 4.2 * len(plot_items)))

for ax, (name, values) in zip(axes, plot_items):
    ax.hist(values, bins=25, color="#4C83C3", edgecolor="white", linewidth=0.6, alpha=0.9)
    ax.set_title(name, pad=10)
    ax.set_xlabel("Значение признака")
    ax.set_ylabel("Количество наблюдений")
    ax.grid(axis="y", alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"Сравнение распределения {feature} до и после масштабирования", fontsize=16, y=1.005)
plt.tight_layout()
plt.show()


In [ ]:
q1, q3 = x_train[feature].quantile([0.25, 0.75])
iqr = q3 - q1
n_out = int(((x_train[feature] < q1 - 1.5 * iqr) | (x_train[feature] > q3 + 1.5 * iqr)).sum())
print(f"Выбросов по правилу IQR в {feature}: {n_out} из {len(x_train)} ({n_out / len(x_train) * 100:.2f}%)")

### 2.2 Вывод по методам масштабирования
Форма гистограммы у всех четырёх методов одинаковая — меняется только шкала по оси X. Это главное, что стоит заметить: масштабирование не исправляет распределение и не убирает выбросы, оно только пересчитывает числа.

- **StandardScaler** — центрирует признак около 0 и делает std = 1. Удобен для линейных моделей, SVM, KNN.
- **MinMaxScaler** — загоняет всё в [0; 1]. Минимум и максимум определяются крайними значениями, поэтому один выброс сжимает всю остальную массу в узкий диапазон.
- **MaxAbsScaler** — делит на максимальный модуль. Нужен для разреженных данных, где важно сохранить нули. Здесь нулей почти нет, так что преимущества нет — по сути получается то же, что MinMax, только без сдвига.
- **RobustScaler** — использует медиану и межквартильный размах, поэтому слабо реагирует на выбросы.

**Что выбрать для `Hours_Studied`:** если выбросов мало (ориентир — меньше 1–2%) и распределение примерно колоколообразное, наиболее подходящий вариант — **StandardScaler**: это стандартный выбор для линейной регрессии, а выбросы здесь слишком редки, чтобы испортить среднее и std. Если же выбросов по IQR заметно больше, лучше взять **RobustScaler**. MinMax здесь менее удачен из-за чувствительности к крайним значениям, MaxAbs — потому что у признака нет разреженности.

### 2.3 Предобработка всех столбцов (после split)
Столбцы бывают разных типов, и каждому типу нужен свой способ:

| Группа | Столбцы | Метод | Почему |
|---|---|---|---|
| Числовые | Hours_Studied, Attendance, Sleep_Hours, Previous_Scores, Tutoring_Sessions, Physical_Activity | SimpleImputer (most frequent) + StandardScaler | Единицы разные (часы, проценты, баллы), приводим к одной шкале; метод из 3.2 |
| Порядковые | Parental_Involvement, Access_to_Resources, Motivation_Level, Family_Income, Teacher_Quality (Low < Medium < High); Parental_Education_Level (High School < College < Postgraduate); Distance_from_Home (Near < Moderate < Far); Peer_Influence (Negative < Neutral < Positive) | SimpleImputer (most frequent) + OrdinalEncoder с заданным порядком | У этих категорий есть реальный порядок, и он важен — код 0, 1, 2 его сохраняет |
| Номинальные / да-нет | Extracurricular_Activities, Internet_Access, Learning_Disabilities, School_Type, Gender | SimpleImputer (most frequent) + OneHotEncoder (для бинарных — один столбец 0/1) | Порядка нет, нельзя вводить искусственное "Public > Private" |

Целевая переменная `Exam_Score` — число, её кодировать не нужно (LabelEncoder вообще нужен только для классов в задачах классификации).

Пропуски заполняются внутри преобразователей после train/test split: статистика заполнения вычисляется только по train и затем применяется к test.

In [ ]:
numeric_cols = ["Hours_Studied", "Attendance", "Sleep_Hours", "Previous_Scores", "Tutoring_Sessions", "Physical_Activity"]

ordinal_map = {
    "Parental_Involvement":     ["Low", "Medium", "High"],
    "Access_to_Resources":      ["Low", "Medium", "High"],
    "Motivation_Level":         ["Low", "Medium", "High"],
    "Family_Income":            ["Low", "Medium", "High"],
    "Teacher_Quality":          ["Low", "Medium", "High"],
    "Parental_Education_Level": ["High School", "College", "Postgraduate"],
    "Distance_from_Home":       ["Near", "Moderate", "Far"],
    "Peer_Influence":           ["Negative", "Neutral", "Positive"],
}
nominal_cols = ["Extracurricular_Activities", "Internet_Access", "Learning_Disabilities", "School_Type", "Gender"]

# страховка: убеждаемся, что заданный порядок покрывает все реальные значения
for col, cats in ordinal_map.items():
    unknown = set(X[col].unique()) - set(cats)
    if unknown:
        print(f"АХТУНГ: в {col} есть значения вне заданного порядка: {unknown}")

covered = set(numeric_cols) | set(ordinal_map) | set(nominal_cols)
print("Столбцы, не попавшие ни в одну группу:", set(X.columns) - covered)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_cols),
        ("ord", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(categories=list(ordinal_map.values()), handle_unknown="use_encoded_value", unknown_value=-1)),
        ]), list(ordinal_map.keys())),
        ("nom", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False)),
        ]), nominal_cols),
    ]
)


X_train_prep = preprocessor.fit_transform(X_train)   # все параметры обучаем только на train
X_test_prep = preprocessor.transform(X_test)          # test преобразуем уже обученными параметрами


feature_names = preprocessor.get_feature_names_out()
X_train_prep = pd.DataFrame(X_train_prep, columns=feature_names, index=X_train.index)
X_test_prep = pd.DataFrame(X_test_prep, columns=feature_names, index=X_test.index)


print("train:", X_train_prep.shape, "| test:", X_test_prep.shape)
X_train_prep.head()


In [ ]:
# быстрая проверка: на train числовые признаки должны иметь среднее ~0 и std ~1, на test — примерно
check = pd.DataFrame({
    "train mean": X_train_prep[[f"num__{c}" for c in numeric_cols]].mean(),
    "train std":  X_train_prep[[f"num__{c}" for c in numeric_cols]].std(),
    "test mean":  X_test_prep[[f"num__{c}" for c in numeric_cols]].mean(),
    "test std":   X_test_prep[[f"num__{c}" for c in numeric_cols]].std(),
}).round(3)
print("Пропусков после предобработки (train/test):", int(X_train_prep.isna().sum().sum()), "/", int(X_test_prep.isna().sum().sum()))
check

Небольшое отличие среднего и std на test от 0 и 1 — это нормально и правильно: параметры взяты по train, test трансформируется "чужой линейкой". Если бы на test тоже получились ровно 0 и 1, это значило бы, что мы пересчитали масштаб заново и допустили утечку.

### 2.4 Итог
Данные очищены, разделены на train/test и приведены к числовому виду без утечки: все параметры (среднее, std, порядок категорий) определены по train и затем применены к test. Следующий шаг — обучение модели регрессии на `X_train_prep`, `y_train` и проверка на `X_test_prep`, `y_test`.